### Import packages

In [ ]:
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as plt

### Define parameters

In [ ]:
N = 24				# number of time steps
I = 2				# number of containers
DELTA_t = 1.0		# hr

ALPHA = 2.61e-5		# MWh capacity lost per MWh throughput
BETA = 2.5			# °C / MWh throughput
GAMMA = 3.0			# °C / hr
ETA = 0.968			# dimensionless
KAPPA = 0.04		# / °C
RHO = 19500			# $ / MWh capacity lost (opportunity)
SIGMA = 200000		# $ / MWh capacity lost (replacement)
A = 0.01			# MW (10 kW)
B = 0.04			# MW (A+Bu = 50 kW at u=1)

T_MIN = -30			# °C
T_REF = 25			# °C
T_MAX = 50			# °C

SOC_MIN = 0.05		# dimensionless
SOC_MAX = 0.95		# dimensionless
Q_NEW = 3.916		# MWh, new container capacity
P_MAX = 0.979		# MW, hard limit

# Day starting values for both containers (arbitrarily chosen)
SOH_0 = np.array([0.98, 0.82])	# dimensionless
SOC_0 = np.array([0.8, 0.4])	# dimensionless

# Day ending values
SOC_N = SOC_0

In [ ]:

# Generate random prices:
# @todo REPLACE WITH REAL-WORLD DATA
PRICE_MIN = 30 # $ / MWh
PRICE_MAX = 250 # $ / MWh
TIME_SHIFT = 3 # hr

AMPLITUDE = (PRICE_MAX - PRICE_MIN) / 2
MU = (PRICE_MAX + PRICE_MIN) / 2
OMEGA = 2 * np.pi / N

def get_price_forecast(k):
	return -1 * AMPLITUDE * np.cos(OMEGA * (k - TIME_SHIFT)) + MU + np.random.normal(loc=0, scale=4)

LAMBDA = np.array([np.round(get_price_forecast(k),2) for k in range(N)])

### Define state & decision variables

In [ ]:
# states: k = 0 -> N, i = 1 -> 2
E = cp.Variable((N + 1, I), nonneg=True)
Q = cp.Variable((N + 1, I), nonneg=True)
T = cp.Variable((N + 1, I), nonneg=True)

# decisions: k = 0 -> N-1, i = 1 -> 2
c = cp.Variable((N, I), nonneg=True)
d = cp.Variable((N, I), nonneg=True)
u = cp.Variable((N, I), nonneg=True)

### Define constraints

In [ ]:
static_constraints = [
	# Initial conditions
	Q[0, :] == Q_NEW * SOH_0,
	E[0, :] == cp.multiply(Q[0, :], SOC_0),
	T[0, :] == T_REF,
	# Dynamics - see below for capacity dynamics
	E[1:, :] == E[:-1, :] + (ETA * c * DELTA_t) - (d * DELTA_t / ETA),
	T[1:, :] == T[:-1, :] + (BETA * (c + d) * DELTA_t) - (GAMMA * u * DELTA_t),
	# Boundary conditions
	Q >= 0,
	E >= Q * SOC_MIN,
	E <= Q * SOC_MAX,
	#T >= T_MIN,
	#T <= T_MAX,
	c >= 0,
	c <= P_MAX,
	d >= 0,
	d <= P_MAX,
	u >= 0,
	u <= 1,
	# Terminal conditions
	E[N, :] >= cp.multiply(Q[N, :], SOC_N)
]

### Successive convex programming

Unfortunately, our temperature-dependent state transition function for capacity is non-convex. 

The following workaround solves the problem iteratively instead:
1. Temperature remains a regular state variable. But _only when calculating the temperature degradation factor_ $\phi(k)$, we initially assume the temperature is $T_\text{ref}$ the whole time.
2. We solve the problem. Temperature changes based on our optimized decisions, but the _degradation_ is as if it stayed static.
3. We plug in our solved values for temperature as a _static_ guess in order to calculate another iteration of $\phi(k)$, and repeat.
4. This converges in just three loops (I tried 10 initially). That is, the temperature state trajectory stops changing with each iteration.

Essentially, while temperature is a state variable, we use the previous iteration's trajectory for temperature as a fixed parameter to calculate $\phi(k)$, making the problem convex.

In [ ]:
T_est = np.full((N, I), T_REF)

for iteration in range(3):
	print(f"Iteration {iteration} starting...")
	phi = 1 + KAPPA * np.maximum(0, T_est - T_REF)
	delta_Q = ALPHA * cp.multiply(phi, c + d) * DELTA_t
	dynamic_constraints = [Q[1:, :] == Q[:-1, :] - delta_Q]

	arbitrage_cost = cp.sum(cp.multiply(LAMBDA[:, None], c - d) * DELTA_t)
	operating_cost = cp.sum(cp.multiply(LAMBDA[:, None], A + B * u) * DELTA_t)
	opportunity_plus_replacement_cost = cp.sum((RHO + SIGMA) * delta_Q)
	objective = cp.Minimize(arbitrage_cost + opportunity_plus_replacement_cost + operating_cost)

	problem = cp.Problem(objective, static_constraints + dynamic_constraints)
	problem.solve(
		solver=cp.OSQP,
		warm_start=True,
		verbose=True,
		max_iter=100000,
		eps_abs=1e-4,
		eps_rel=1e-4,
	)
	if problem.status == "optimal":
		T_est = T.value[:-1, :]
		print(f"Iteration {iteration}: Total Cost = {problem.value:.2f}")
	else:
		print("Solver failed to find an optimal solution.")
		break


### Plot

In [ ]:
fig, axes = plt.subplots(4, figsize=(8,16))
K = np.array(range(N+1))

color1 = 'tab:blue'
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Stored Energy (MWh)', color=color1, fontweight='bold')
axes[0].plot(K, E.value[:,0], color=color1, linewidth=2, label='Container 1', marker='o')
axes[0].plot(K, E.value[:,1], color='red', linewidth=2, label='Container 2', marker='o')
axes[0].tick_params(axis='y', labelcolor=color1)
axes[0].set_ylim(0, Q_NEW + 1) # Slightly above max capacity
axes[0].set_title('BESS Energy Dispatch vs. Market Price')
axes[0].legend()

ax2 = axes[0].twinx()
color2 = 'tab:green'
ax2.set_ylabel('RTM Price ($/MWh)', fontweight='bold')
ax2.step(K[:-1], LAMBDA, color=color2, where='post', alpha=0.6, label='Price')
ax2.tick_params(axis='y')

axes[1].plot(K, Q.value[:,0] - Q.value[0,0], label="Container 1", color=color1, marker='o')
axes[1].plot(K, Q.value[:,1] - Q.value[0,1], label="Container 2", color='red', marker='o')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Change in Capacity (MWh)', fontweight='bold')
axes[1].set_title("Change in Container Capacity")
axes[1].legend()

axes[2].plot(K, T.value[:,0], label="Container 1", color=color1, marker='o')
axes[2].plot(K, T.value[:,1], label="Container 2", color='red', marker='o')
axes[2].set_xlabel('Hour of Day')
axes[2].set_ylabel('Temperature', fontweight='bold')
axes[2].set_title("Container Temperature")
axes[2].legend()

axes[3].plot(K[:-1], c.value[:,0], marker='o', color='blue', label="C1 Charging")
axes[3].plot(K[:-1], -d.value[:,0], marker='o', color='blue', linestyle='--', label="C1 Charging")
axes[3].plot(K[:-1], c.value[:,1], marker='o', color='red', label="C2 Discharging")
axes[3].plot(K[:-1], -d.value[:,1], marker='o', color='red', linestyle='--', label="C2 Discharging")
axes[3].set_xlabel("Hour of Day")
axes[3].set_ylabel("Power setpoint (MW)")
axes[3].legend()
axes[3].set_title("Power setpoints")

plt.grid(axis='x', linestyle='--')
plt.tight_layout()
plt.show()
